# PathQ-Former v2 — 5-fold CV on Colab (GPU)

Generic notebook for any cancer type. Set `CANCER` and `CONFIG` in the first code cell.

* **Runtime > Change runtime type > T4 GPU** (or better).
* Add your HuggingFace token as a Colab secret named `HF_TOKEN` (gated dataset `MahmoodLab/UNI2-h-features`).
* Code comes from a `PathQFormer_src_<sha>.zip` on Drive (`python scripts/package_src.py` locally) or from a git URL.
* Everything resumes: rerun all cells after a disconnect; finished folds are skipped.

In [ ]:
CANCER = 'blca'            # blca | brca | stad | coadread | hnsc
CONFIG = 'configs/blca_hybrid_v2.yaml'
GIT_URL = ''               # optional, e.g. 'https://github.com/<you>/PathQFormer.git'; leave empty to use the Drive zip
DRIVE_ROOT = '/content/drive/MyDrive/PathQFormer'

from google.colab import drive
drive.mount('/content/drive')
import os, glob
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs_v2/hybrid'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ---- code -------------------------------------------------------------
import subprocess, os, glob
os.chdir('/content')
if GIT_URL:
    if not os.path.exists('PathQFormer'):
        !git clone -q {GIT_URL} PathQFormer
else:
    zips = sorted(glob.glob(f'{DRIVE_ROOT}/PathQFormer_src_*.zip'), key=os.path.getmtime)
    assert zips, f'upload PathQFormer_src_<sha>.zip to {DRIVE_ROOT} (python scripts/package_src.py)'
    !unzip -qo "{zips[-1]}" -d /content/
os.chdir('/content/PathQFormer')
!pip install -q scikit-survival lifelines h5py pyyaml huggingface_hub
!python -m pytest -q tests || true

In [ ]:
# ---- embeddings -------------------------------------------------------
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
EMB_DIR = '/content/embeddings'
!python scripts/download_embeddings.py --cancer {CANCER.upper()} --dest {EMB_DIR}
print(len(glob.glob(f'{EMB_DIR}/**/*.h5', recursive=True)), 'slides')

In [ ]:
# ---- train (resumable) ------------------------------------------------
!python -m src.training.train --config {CONFIG} --cancer_type {CANCER} \n    --set embeddings_dir={EMB_DIR} output_dir={OUTPUT_DIR} num_workers=2

In [ ]:
# ---- results ----------------------------------------------------------
from IPython.display import Markdown, display
display(Markdown(open(f'{OUTPUT_DIR}/{CANCER}/summary.md').read()))
!python scripts/analyze_run.py {OUTPUT_DIR}/{CANCER} --attention --figdir {DRIVE_ROOT}/figures/{CANCER}